# Cell marker positivity — training

A CNN that decides whether one segmented cell is positive for one IMC marker.

The input to the network is a crop centred on the cell:

* the target marker channel plus its neighbouring metal channels (`N_SIDE` on each side),
* the binary mask of the target cell,
* optionally the binary mask of all *other* cells in the crop (`USE_OTHER_CELLS`).

Every region comes with the same cells drawn at a range of sizes, from the bare nucleus
up to ten pixels of growth. Each sample picks one of those sizes at random, so the model
cannot learn to read positivity off how large a cell happens to be drawn.

Nothing is written to disk. Only the marker volumes and one label map per size are held
in memory; the crops are cut out of them on the fly, every iteration.

In [ ]:
import numpy as np
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import ndimage
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

# --- data -----------------------------------------------------------------
DATA_DIR = "TrainData"

# Each region is a marker stack and the same cells drawn at a range of sizes.
DATASETS = [
    ("EMMA1.nii", [
        "EMMA1_inference_cell_nuclei_mask_corrected_labled.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated1.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated2.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated3.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated4.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated5.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated6.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated7.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated8.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated9.nii",
        "EMMA1_inference_cell_nuclei_mask_corrected_labled_dilated10.nii",
    ]),
    ("20231201_713Lung_E_3_20231201-154853-900840_Q013_markers_channels_zstack.nii.gz", [
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated1.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated2.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated3.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated4.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated5.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated6.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated7.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated8.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated9.nii",
        "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_nuclei_mask_corrected_labled_dilated10.nii",
    ]),
]

# Values used in the positivity masks.
NEGATIVE, POSITIVE, UNKNOWN = 1, 2, 3

# --- model input ----------------------------------------------------------
CROP = 64
N_SIDE = 2                 # neighbouring metal channels on each side
USE_OTHER_CELLS = True     # add the mask of the surrounding cells

N_MARKERS = 2 * N_SIDE + 1
N_CHANNELS = N_MARKERS + 1 + int(USE_OTHER_CELLS)

# --- augmentation ---------------------------------------------------------
INTENSITY_GAIN = (0.95, 1.05)   # random gain on the marker channels
INTENSITY_OFFSET = 0.02         # random offset, as a fraction of the crop's mean

SPILL_CHANCE = 0.3              # how often spillover between metals is simulated
SPILL_STRENGTH = (0.02, 0.20)   # size of the copy relative to the channel it comes from
SPILL_DECAY = 0.5               # how much weaker it gets per channel of distance
SPILL_NOISE = 0.3               # how rough the copy is

# --- training -------------------------------------------------------------
SEED = 42
VAL_FRAC = 0.2
BATCH_SIZE = 512
EPOCHS = 100
BASE_LR, MIN_LR, CYCLE_LENGTH = 1e-3, 1e-6, 20

print(f"Input: {CROP}x{CROP}x{N_CHANNELS}")

## Load the regions and index the labelled cells

Each region is a marker stack `(H, W, Z)` and, at every size, a positivity mask of the
same shape in which every cell is filled with 1 (negative), 2 (positive) or 3 (unknown)
for each marker channel.

The cell segmentation is the same for every marker, so the cells are labelled once per
size. Each size is then renumbered to carry the smallest mask's numbering, so one cell
can be found at any size. Every cell is uniform within a channel, so one representative
pixel is enough to read off its values for all channels at once. Cells that are unknown
in a channel are simply left out of the index.

In [ ]:
class Region:
    """One imaged region held in memory, with its cells drawn at every size."""

    def __init__(self, markers, cells, centres):
        self.markers = markers   # (H, W, Z) float32, the marker stack
        self.cells = cells       # per size, (H, W) int32 cell numbers
        self.centres = centres   # per size, (n_cells, 2) int32 cell centres


def align_to_smallest(smallest, cells):
    """Give a grown mask the cell numbers the smallest mask uses.

    A connected component analysis numbers cells in the order it meets them, and that
    order changes as they grow. A grown cell covers the cell it grew from, so the cell
    of the smallest mask it shares most pixels with says which one it is.
    """
    span = int(cells.max()) + 1
    both = (smallest > 0) & (cells > 0)

    pair, count = np.unique(smallest[both].astype(np.int64) * span + cells[both],
                            return_counts=True)
    order = np.argsort(count)                # smallest overlap first, so the largest wins

    renumber = np.zeros(span, dtype=np.int32)
    renumber[(pair % span)[order]] = (pair // span)[order]

    return renumber[cells]


def load_region(image_file, mask_files):
    """Return the region at every size and, per cell and channel, its positivity."""
    markers = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{image_file}"))
    markers = markers.get_fdata().astype(np.float32)

    cells_at_size, centres_at_size, values = [], [], None

    for mask_file in mask_files:
        positivity = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{mask_file}"))
        positivity = positivity.get_fdata().astype(np.int16)

        if markers.shape != positivity.shape:
            raise ValueError(f"{mask_file}: {positivity.shape} against {markers.shape}")

        cells, n_cells = ndimage.label(positivity[:, :, 0] > 0)
        cells = cells.astype(np.int32)

        if values is None:
            # The smallest mask fixes the numbering, and the values every size shares.
            representative = np.zeros(n_cells + 1, dtype=np.int64)
            representative[cells.ravel()] = np.arange(cells.size)
            values = positivity.reshape(-1, positivity.shape[2])[representative[1:]]
        else:
            cells = align_to_smallest(cells_at_size[0], cells)

        cells_at_size.append(cells)
        centres_at_size.append(np.array(ndimage.center_of_mass(
            cells > 0, cells, np.arange(1, len(values) + 1))).astype(np.int32))

    return Region(markers, cells_at_size, centres_at_size), values


regions = []
index = []    # columns: region, z, cell number
labels = []   # 1.0 = positive

for region_id, (image_file, mask_files) in enumerate(DATASETS):
    region, values = load_region(image_file, mask_files)
    regions.append(region)

    labelled = (values == NEGATIVE) | (values == POSITIVE)
    cell, z = np.nonzero(labelled)

    index.append(np.column_stack([np.full(cell.size, region_id), z, cell + 1]))
    labels.append(values[cell, z] == POSITIVE)

    print(f"{image_file[:40]:40s} {region.markers.shape}  "
          f"{len(region.cells):2d} sizes  {values.shape[0]:5d} cells  "
          f"{cell.size:6d} labelled  {labelled.size - cell.size:5d} unknown")

index = np.concatenate(index).astype(np.int32)
labels = np.concatenate(labels).astype(np.float32)

memory = sum(r.markers.nbytes + sum(c.nbytes for c in r.cells) for r in regions)
print(f"\nTotal: {len(index)} samples, {labels.mean():.1%} positive, "
      f"{memory / 1e6:.0f} MB in memory")

## Cutting a sample out of a region

The crop is centred on the cell but kept inside the image, exactly as the inference
notebook does it.

Near the ends of the stack some of the neighbouring channels do not exist. Those are
left empty rather than repeating the edge channel: a copy of the target channel would
be perfectly correlated with it, which is exactly what spillover looks like, whereas
an empty channel simply says that there is no such neighbour.

Every sample takes its masks from the nucleus mask or from any of the grown ones, picked
at random, which is what keeps the model from reading positivity off how large a cell
happens to be drawn.

Samples are augmented in four further ways. The crop is rotated and flipped, and the order of
the marker channels is reversed, which makes the model indifferent to whether the stack
was written in ascending or descending order; the target channel sits in the middle, so
only its neighbours swap sides. The intensities then get a small random gain and offset,
and, some of the time, a weakened noisy copy of one channel is bled into the others to
imitate spillover between metals. The two mask channels are never altered, and neither
are the empty channels beyond the ends of the stack.

In [ ]:
def make_sample(region_id, z, cell_id, size):
    region = regions[region_id]
    height, width, n_z = region.markers.shape
    half = CROP // 2

    centre_y, centre_x = region.centres[size][cell_id - 1]

    y = min(max(centre_y - half, 0), height - CROP)
    x = min(max(centre_x - half, 0), width - CROP)

    cells = region.cells[size][y:y + CROP, x:x + CROP]

    # Channels beyond either end of the stack do not exist and stay empty.
    wanted = np.arange(z - N_SIDE, z + N_SIDE + 1)
    exists = (wanted >= 0) & (wanted < n_z)

    markers = np.zeros((CROP, CROP, N_MARKERS), dtype=np.float32)
    markers[..., exists] = region.markers[y:y + CROP, x:x + CROP, wanted[exists]]

    planes = [markers, (cells == cell_id)[..., None]]
    if USE_OTHER_CELLS:
        planes.append(((cells > 0) & (cells != cell_id))[..., None])

    # concatenating the float32 markers with the boolean masks already gives float32
    return np.concatenate(planes, axis=-1, dtype=np.float32)


def add_spillover(markers, level, rng):
    """Bleed a weakened, noisy copy of one channel into the others.

    Spillover between metals falls off with distance, so the copy is halved for every
    further channel away from its source. Showing the network what this looks like lets
    it tell a faint echo of a strong neighbour apart from real expression.
    """
    source = rng.integers(N_MARKERS)

    distance = np.abs(np.arange(N_MARKERS) - source)
    weight = rng.uniform(*SPILL_STRENGTH) * SPILL_DECAY ** (distance - 1)
    weight[source] = 0.0
    weight[level == 0] = 0.0                  # a channel that is not there gets nothing

    noise = 1.0 + SPILL_NOISE * rng.standard_normal(markers.shape, dtype=np.float32)

    return markers + markers[..., source, None] * (weight.astype(np.float32) * noise)


def jitter_intensity(markers, level, rng):
    """A small gain and offset per channel, too small to change a cell's positivity.

    The offset follows each channel's own signal level, both because the regions differ
    in absolute counts by more than a factor of ten, and because it keeps the empty
    channels beyond the ends of the stack empty.
    """
    gain = rng.uniform(*INTENSITY_GAIN, size=N_MARKERS).astype(np.float32)
    offset = rng.uniform(-1.0, 1.0, N_MARKERS).astype(np.float32) * INTENSITY_OFFSET * level

    return markers * gain + offset


def augment(sample, rng):
    """Turn and flip the crop, reverse the channel order, then disturb the intensities."""
    sample = np.rot90(sample, rng.integers(4), axes=(0, 1))
    if rng.random() < 0.5:
        sample = sample[:, ::-1]
    if rng.random() < 0.5:
        sample = np.concatenate(
            [sample[..., N_MARKERS - 1::-1], sample[..., N_MARKERS:]], axis=-1)

    markers = np.ascontiguousarray(sample[..., :N_MARKERS])

    # How much signal each channel carries. A channel beyond the end of the stack is
    # empty and stays at zero, which keeps both augmentations off it.
    level = markers.mean(axis=(0, 1))

    if rng.random() < SPILL_CHANCE:
        markers = add_spillover(markers, level, rng)
    markers = jitter_intensity(markers, level, rng)

    np.maximum(markers, 0.0, out=markers)      # the offset can push a channel negative

    return np.concatenate([markers, sample[..., N_MARKERS:]], axis=-1, dtype=np.float32)


class CellSequence(tf.keras.utils.Sequence):
    def __init__(self, rows, shuffle, augment_samples):
        self.rows = rows
        self.shuffle = shuffle
        self.augment_samples = augment_samples
        self.rng = np.random.default_rng(SEED)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.rows) / BATCH_SIZE))

    def on_epoch_end(self):
        self.order = self.rng.permutation(self.rows) if self.shuffle else self.rows

    def __getitem__(self, batch):
        rows = self.order[batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE]
        x = np.empty((len(rows), CROP, CROP, N_CHANNELS), dtype=np.float32)

        for i, row in enumerate(rows):
            region_id, z, cell_id = index[row]

            # the nucleus mask or any of the grown ones, chosen at random
            size = self.rng.integers(len(regions[region_id].cells))

            sample = make_sample(region_id, z, cell_id, size)
            x[i] = augment(sample, self.rng) if self.augment_samples else sample

        return x, labels[rows]


shuffled = np.random.default_rng(SEED).permutation(len(index))
n_val = int(len(index) * VAL_FRAC)

train_gen = CellSequence(shuffled[n_val:], shuffle=True, augment_samples=True)
val_gen = CellSequence(shuffled[:n_val], shuffle=False, augment_samples=False)

print(f"Train: {len(train_gen.rows)} samples, val: {len(val_gen.rows)} samples")

## Check a sample

Every input channel of one random cell, so it is easy to see that the marker
channels, the cell mask and the surrounding cells line up.

In [ ]:
row = np.random.default_rng(0).integers(len(index))
region_id, z, cell_id = index[row]
sample = make_sample(region_id, z, cell_id, 0)

names = [f"marker z{z:+d}" for z in range(-N_SIDE, N_SIDE + 1)] + ["this cell"]
if USE_OTHER_CELLS:
    names.append("other cells")

fig, axes = plt.subplots(1, N_CHANNELS, figsize=(3 * N_CHANNELS, 3.4))
for ax, name, plane in zip(axes, names, np.moveaxis(sample, -1, 0)):
    ax.imshow(plane, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

fig.suptitle(f"region {region_id}, channel {z}, cell {cell_id} — "
             f"{'POSITIVE' if labels[row] else 'NEGATIVE'}")
plt.tight_layout()
plt.show()

# the same cell at every size, which is what a sample picks from at random
sizes = range(len(regions[region_id].cells))
fig, axes = plt.subplots(1, len(sizes), figsize=(1.7 * len(sizes), 2.2))
for size, ax in zip(sizes, axes):
    ax.imshow(make_sample(region_id, z, cell_id, size)[..., N_MARKERS], cmap="gray")
    ax.set_title(f"+{size}", fontsize=9)
    ax.axis("off")
fig.suptitle("the same cell, grown one pixel at a time")
plt.tight_layout()
plt.show()

## Model

In [ ]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(filters, kernel_size, strides=strides,
                      padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)


def make_marker_cnn(input_shape):
    inputs = layers.Input(shape=input_shape)

    c = 32
    x = conv_bn_relu(inputs, c)
    x = conv_bn_relu(x, c)
    x = conv_bn_relu(x, c)
    x = conv_bn_relu(x, c)

    x = conv_bn_relu(x, 2 * c, strides=2)
    x = conv_bn_relu(x, 4 * c, strides=2)
    x = conv_bn_relu(x, 8 * c, strides=2)

    x = layers.Conv2D(1, 1)(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Activation("sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


model = make_marker_cnn((CROP, CROP, N_CHANNELS))
model.summary()

## Train

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)


def cosine_annealing_with_restarts(epoch, lr):
    t = (epoch % CYCLE_LENGTH) / CYCLE_LENGTH
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + np.cos(np.pi * t))


callbacks = [
    tf.keras.callbacks.LearningRateScheduler(cosine_annealing_with_restarts),
    tf.keras.callbacks.CSVLogger("training_log.csv", append=True),
    tf.keras.callbacks.ModelCheckpoint("checkpoints/marker_cnn_epoch_{epoch:03d}.h5"),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    workers=8,
    use_multiprocessing=False,
)

model.save("marker_cnn_final.h5")
print("Training complete.")

## Training curves

In [ ]:
for key in ["loss", "accuracy", "auc"]:
    plt.figure()
    plt.plot(history.history[key], label=f"train {key}")
    plt.plot(history.history[f"val_{key}"], label=f"val {key}")
    plt.xlabel("epoch")
    plt.ylabel(key)
    plt.legend()
    plt.grid(True)
    plt.show()

## Architecture figure (optional)

Needs `pip install visualkeras pillow`.

In [ ]:
import visualkeras
from PIL import Image
from IPython.display import Image as IPyImage


def text_callable(layer_index, layer):
    shape = [s for s in layer.output.shape if s is not None]
    text = "x".join(str(s) for s in shape[:-1]) + f"\n{shape[-1]}" if shape else "?"
    return f"{text}\n{layer.name}", bool(layer_index % 2)


image = visualkeras.layered_view(
    model, legend=True, show_dimension=True, spacing=15,
    draw_volume=True, scale_xy=0.8, text_callable=text_callable,
)
image = image.resize((int(image.width * 0.6), image.height), Image.BILINEAR)
image.save("marker_cnn_3d.png")

IPyImage("marker_cnn_3d.png")